In [ ]:
from scipy import stats
import numpy as np



#예시 값
logpress = [1, 1, 0, 1]    

streaming = [1, 0, 1, 0]
snapkv    = [1, 0, 0, 1]
pyramid   = [0, 1, 0, 1]
h2o       = [1, 1, 0, 0]

baseline = {"snapkv" : snapkv, "streaming" : streaming, "h2o" : h2o, "pyramid": pyramid}

# 원인 줄 메타데이터 (층 나누기용)
cause    = [2, 5, 7, 8]                  # 원인 줄 번호
is_error = {2: True, 5: False, 7: False, 8: True}
pos      = {2: 0.30, 5: 0.55, 7: 0.80, 8: 0.90}

keep = {2: True, 5: False, 7: True, 8: True}

overall = [0,1,2,3]
non_error = [i for i in range(4) if not is_error[cause[i]]]
middle = [i for i in range (4) if 0.25 <= pos[cause[i]] <= 0.75]

strata = {"overall" : overall, "non_error" : non_error, "middle" : middle}


In [ ]:
# 각 방법이 층별로 원인줄을 몇프로 살렸나?
def preservation(result, target):
    return sum(1 for c in target if result[c]) / len(target)

print("-- 층별 보존율 --")
print(f"{'method' : 10}", *[f"{s:>10}" for s in strata])
for name, res in {"logpress" : logpress, **baseline}.items():
    rates = [preservation(res, t) for t in strata.values()]
    print(f"{name:10}", *[f"{r:>10.2f}" for r in rates])

In [ ]:
def mcnemar(x, y, idx):
    p = q = 0
    for i in idx:
        if x[i] != y[i]:
            if x[i] == 1:
                p += 1
            else:
                q += 1
    n = p+q
    p_value = 1.0 if n == 0 else min(2* stats.binom.cdf(min(p,q), n, 0.5), 1.0)
    return p,q, p_value

results = []
for b_name, base in baseline.items():
    for s_name, idx in strata.items():
        p,q,p_value = mcnemar(logpress, base, idx)
        results.append([b_name, s_name, p, q, p_value])
        print(f"{b_name:10} {s_name:10} p = {p} q = {q} p_value = {round(p_value,3)}")


# "어떤 사건이 성공할 확률이 50%($0.5$)인 시행을 총 $n$번 했을 때, 성공 횟수가 최소값인 min(b, c)번 이하가 될 확률"
# 이항분포의 누적분포함수 계산


snapkv     overall    p = 1 q = 0 p_value = 1.0
snapkv     non_error  p = 1 q = 0 p_value = 1.0
snapkv     middle     p = 1 q = 0 p_value = 1.0
streaming  overall    p = 2 q = 1 p_value = 1.0
streaming  non_error  p = 1 q = 1 p_value = 1.0
streaming  middle     p = 1 q = 0 p_value = 1.0
h2o        overall    p = 1 q = 0 p_value = 1.0
h2o        non_error  p = 0 q = 0 p_value = 1.0
h2o        middle     p = 0 q = 0 p_value = 1.0
pyramid    overall    p = 1 q = 0 p_value = 1.0
pyramid    non_error  p = 0 q = 0 p_value = 1.0
pyramid    middle     p = 1 q = 0 p_value = 1.0


In [ ]:
#p값 BH보정 : Tier-1 pass/fail
#임의의 p값들
p_values = [0.001, 0.008, 0.02, 0.03, 0.04, 0.045,
         0.3, 0.4, 0.5, 0.6, 0.8, 0.9]

def bh_fdr(p_values, alpha = 0.05):
    p_values = np.array(p_values)
    m = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adj = ranked * m / (np.arange(m) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out

p_adj = bh_fdr([r[4] for r in results])

for before, after in zip(p_values, p_adj):
    print(f"보정 전 {before:.3f} -> 보정 후 {after:.3f}")

print(f"{'baseline':10}{'stratum':10}{'b':>3}{'c':>3}{}")

보정 전 0.001 -> 보정 후 0.012
보정 전 0.008 -> 보정 후 0.048
보정 전 0.020 -> 보정 후 0.080
보정 전 0.030 -> 보정 후 0.090
보정 전 0.040 -> 보정 후 0.090
보정 전 0.045 -> 보정 후 0.090
보정 전 0.300 -> 보정 후 0.514
보정 전 0.400 -> 보정 후 0.600
보정 전 0.500 -> 보정 후 0.667
보정 전 0.600 -> 보정 후 0.720
보정 전 0.800 -> 보정 후 0.873
보정 전 0.900 -> 보정 후 0.900
